# Notebook 01 — SQL Server → MinIO Landing Zone (CSV)

Extrai **todas as 11 tabelas** do SeguroDB e grava cada uma como arquivo CSV no bucket `landing-zone` do MinIO.

**Fluxo:**
```
SQL Server (SeguroDB) ──pyodbc──► pandas DataFrame ──boto3──► MinIO s3://landing-zone/{tabela}.csv
```

## 1. Imports e configuração

In [ ]:
import os
import io
import pyodbc
import pandas as pd
import boto3
from botocore.client import Config
from dotenv import load_dotenv

load_dotenv()

# SQL Server
DB_SERVER = os.getenv('DB_SERVER', 'localhost')
DB_PORT   = os.getenv('DB_PORT', '1433')
DB_USER   = os.getenv('DB_USER', 'sa')
DB_PASS   = os.getenv('DB_PASSWORD', 'Sa@123456')
DB_NAME   = os.getenv('DB_DATABASE', 'SeguroDB')
DB_SCHEMA = os.getenv('DB_SCHEMA', 'dbo')

# MinIO
MINIO_ENDPOINT    = os.getenv('MINIO_ENDPOINT', 'http://localhost:9020')
MINIO_ACCESS_KEY  = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET_KEY  = os.getenv('MINIO_SECRET_KEY', 'minioadmin')
LANDING_BUCKET    = os.getenv('MINIO_LANDING_BUCKET', 'landing-zone')

CONN_STR = (
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={DB_SERVER},{DB_PORT};"
    f"DATABASE={DB_NAME};"
    f"UID={DB_USER};"
    f"PWD={DB_PASS};"
    f"TrustServerCertificate=yes;"
)

print(f'SQL Server: {DB_SERVER}:{DB_PORT}/{DB_NAME}')
print(f'MinIO: {MINIO_ENDPOINT} → bucket: {LANDING_BUCKET}')

## 2. Conectar ao MinIO e criar bucket landing-zone

In [ ]:
s3 = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

# Cria o bucket se não existir
existing_buckets = [b['Name'] for b in s3.list_buckets()['Buckets']]
if LANDING_BUCKET not in existing_buckets:
    s3.create_bucket(Bucket=LANDING_BUCKET)
    print(f'Bucket criado: {LANDING_BUCKET}')
else:
    print(f'Bucket já existe: {LANDING_BUCKET}')

## 3. Descobrir tabelas do banco

In [ ]:
conn = pyodbc.connect(CONN_STR)

query_tabelas = f"""
    SELECT TABLE_NAME
    FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_TYPE = 'BASE TABLE'
      AND TABLE_SCHEMA = '{DB_SCHEMA}'
    ORDER BY TABLE_NAME
"""

df_tabelas = pd.read_sql(query_tabelas, conn)
tabelas = df_tabelas['TABLE_NAME'].tolist()

print(f'Tabelas encontradas no {DB_NAME}: {len(tabelas)}')
for t in tabelas:
    print(f'  - {t}')

conn.close()

## 4. Extrair cada tabela e enviar para landing-zone

In [ ]:
conn = pyodbc.connect(CONN_STR)

resultados = []

for tabela in tabelas:
    # Lê a tabela completa
    df = pd.read_sql(f"SELECT * FROM {DB_SCHEMA}.{tabela}", conn)

    # Serializa para CSV em memória
    buffer = io.StringIO()
    df.to_csv(buffer, index=False, encoding='utf-8')
    csv_bytes = buffer.getvalue().encode('utf-8')

    # Envia para MinIO
    key = f"{tabela}.csv"
    s3.put_object(
        Bucket=LANDING_BUCKET,
        Key=key,
        Body=csv_bytes,
        ContentType='text/csv'
    )

    size_kb = len(csv_bytes) / 1024
    resultados.append({'tabela': tabela, 'registros': len(df), 'tamanho_kb': round(size_kb, 2)})
    print(f'  {tabela:15s} → {len(df):5d} registros → {size_kb:.1f} KB → s3://{LANDING_BUCKET}/{key}')

conn.close()

df_resultado = pd.DataFrame(resultados)
print(f'\nTotal: {df_resultado["registros"].sum()} registros, {df_resultado["tamanho_kb"].sum():.1f} KB')

## 5. Validação — listar arquivos no landing-zone

In [ ]:
response = s3.list_objects_v2(Bucket=LANDING_BUCKET)
objetos = response.get('Contents', [])

print(f'=== Arquivos no bucket "{LANDING_BUCKET}" ===')
for obj in objetos:
    print(f'  {obj["Key"]:30s}  {obj["Size"]/1024:.1f} KB  {obj["LastModified"].strftime("%Y-%m-%d %H:%M")}')

print(f'\nTotal: {len(objetos)} arquivos')